Linear Regression from Scratch

In [1]:
import random
import math
random.seed(42)
TRUE_W = 3.0
TRUE_B = 7.0
N_SAMPLES = 100

X = [random.uniform(0, 10) for _ in range(N_SAMPLES)]
y = [TRUE_W * x + TRUE_B + random.gauss(0, 2.0) for x in X]

print(f"Generated {N_SAMPLES} samples")
print(f"True relationship: y = {TRUE_W}x + {TRUE_B} (+ noise)")
print(f"First 5 points: {[(round(X[i], 2), round(y[i], 2)) for i in range(5)]}")


Generated 100 samples
True relationship: y = 3.0x + 7.0 (+ noise)
First 5 points: [(6.39, 29.37), (0.25, 7.98), (2.75, 14.22), (2.23, 11.44), (7.36, 28.79)]


In [12]:
class LinearRegression:
    def __init__(self,lr=0.01):
        self.w = 0.0
        self.b = 0.0
        self.lr = lr
        self.history = []
        
    def predict(self,X):
        return [self.w*x+self.b for x in X]

    def compute_loss(self,X,y):
        predictions = self.predict(X)
        n = len(y)
        mse_loss = sum((pred - actual)**2 for pred,actual in zip(predictions,y))/n
        return mse_loss
        
    def compute_gradients(self,X,y):
        predictions = self.predict(X)
        n = len(y)
        dw = (2/n)*sum((pred-actual) * x for pred,actual, x in zip(predictions,y,X))
        db = (2/n)*sum(pred-actual for pred,actual in zip(predictions,y))
        return dw,db
        
    def fit(self,X,y,epochs=1000,print_every=200):
        for epoch in range(epochs):
            dw,db = self.compute_gradients(X,y)
            self.w -= self.lr * dw
            self.b -= self.lr * db
            loss = self.compute_loss(X,y)
            self.history.append(loss)
            if epoch % print_every == 0:
                print(f" Epoch {epoch:4d} | Loss: {loss:.4f} | w: {self.w:.4f} | b: {self.b:.4f}")
        return self
            
    def r_sqr(self,X,y):
        predictions = self.predict(X)
        y_mean = sum(y)/len(y)
        ss_res = sum((actual-pred)**2 for actual,pred in zip(y,predictions))
        ss_total = sum((actual-y_mean)**2 for actual in y)
        return 1 - (ss_res/ss_total)
        
print("=== Training Linear Regression (Gradient Descent) ===")
model = LinearRegression(lr=0.005)
model.fit(X, y, epochs=1000, print_every=200)
print(f"\nLearned: y = {model.w:.4f}x + {model.b:.4f}")
print(f"True:    y = {TRUE_W}x + {TRUE_B}")
print(f"R-squared: {model.r_sqr(X, y):.4f}")     
        

=== Training Linear Regression (Gradient Descent) ===
 Epoch    0 | Loss: 263.0919 | w: 1.3073 | b: 0.2174
 Epoch  200 | Loss: 7.4251 | w: 3.6454 | b: 3.2610
 Epoch  400 | Loss: 4.9453 | w: 3.4099 | b: 4.8006
 Epoch  600 | Loss: 4.0920 | w: 3.2717 | b: 5.7038
 Epoch  800 | Loss: 3.7984 | w: 3.1906 | b: 6.2336

Learned: y = 3.1433x + 6.5432
True:    y = 3.0x + 7.0
R-squared: 0.9566


In [13]:
class LinearRegressionNormal:
    def __init__(self):
        self.w = 0.0
        self.b = 0.0

    def fit(self, X, y):
        n = len(X)
        x_mean = sum(X) / n
        y_mean = sum(y) / n
        numerator = sum((X[i] - x_mean) * (y[i] - y_mean) for i in range(n))
        denominator = sum((X[i] - x_mean) ** 2 for i in range(n))
        self.w = numerator / denominator
        self.b = y_mean - self.w * x_mean
        return self

    def predict(self, X):
        return [self.w * x + self.b for x in X]

    def r_squared(self, X, y):
        predictions = self.predict(X)
        y_mean = sum(y) / len(y)
        ss_res = sum((actual - pred) ** 2 for actual, pred in zip(y, predictions))
        ss_tot = sum((actual - y_mean) ** 2 for actual in y)
        return 1 - (ss_res / ss_tot)


print("\n=== Normal Equation (Closed-Form) ===")
model_normal = LinearRegressionNormal()
model_normal.fit(X, y)
print(f"Learned: y = {model_normal.w:.4f}x + {model_normal.b:.4f}")
print(f"R-squared: {model_normal.r_squared(X, y):.4f}")


=== Normal Equation (Closed-Form) ===
Learned: y = 3.0756x + 6.9853
R-squared: 0.9572


In [14]:
class MultipleLinearRegression:
    def __init__(self, n_features, learning_rate=0.01):
        self.weights = [0.0] * n_features
        self.bias = 0.0
        self.lr = learning_rate
        self.cost_history = []

    def predict_single(self, x):
        return sum(w * xi for w, xi in zip(self.weights, x)) + self.bias

    def predict(self, X):
        return [self.predict_single(x) for x in X]

    def compute_cost(self, X, y):
        predictions = self.predict(X)
        n = len(y)
        return sum((pred - actual) ** 2 for pred, actual in zip(predictions, y)) / n

    def fit(self, X, y, epochs=1000, print_every=200):
        n = len(y)
        n_features = len(X[0])
        for epoch in range(epochs):
            predictions = self.predict(X)
            errors = [pred - actual for pred, actual in zip(predictions, y)]
            for j in range(n_features):
                grad = (2 / n) * sum(errors[i] * X[i][j] for i in range(n))
                self.weights[j] -= self.lr * grad
            grad_b = (2 / n) * sum(errors)
            self.bias -= self.lr * grad_b
            cost = self.compute_cost(X, y)
            self.cost_history.append(cost)
            if epoch % print_every == 0:
                print(f"  Epoch {epoch:4d} | Cost: {cost:.4f}")
        return self

    def r_squared(self, X, y):
        predictions = self.predict(X)
        y_mean = sum(y) / len(y)
        ss_res = sum((actual - pred) ** 2 for actual, pred in zip(y, predictions))
        ss_tot = sum((actual - y_mean) ** 2 for actual in y)
        return 1 - (ss_res / ss_tot)


random.seed(42)
N = 100
X_multi = []
y_multi = []
for _ in range(N):
    size = random.uniform(500, 3000)
    bedrooms = random.randint(1, 5)
    age = random.uniform(0, 50)
    price = 50 * size + 10000 * bedrooms - 1000 * age + 50000 + random.gauss(0, 20000)
    X_multi.append([size, bedrooms, age])
    y_multi.append(price)


def standardize(X):
    n_features = len(X[0])
    means = [sum(X[i][j] for i in range(len(X))) / len(X) for j in range(n_features)]
    stds = []
    for j in range(n_features):
        variance = sum((X[i][j] - means[j]) ** 2 for i in range(len(X))) / len(X)
        stds.append(variance ** 0.5)
    X_scaled = []
    for i in range(len(X)):
        row = [(X[i][j] - means[j]) / stds[j] if stds[j] > 0 else 0 for j in range(n_features)]
        X_scaled.append(row)
    return X_scaled, means, stds


y_mean_val = sum(y_multi) / len(y_multi)
y_std_val = (sum((yi - y_mean_val) ** 2 for yi in y_multi) / len(y_multi)) ** 0.5
y_scaled = [(yi - y_mean_val) / y_std_val for yi in y_multi]

X_scaled, x_means, x_stds = standardize(X_multi)

print("\n=== Multiple Linear Regression (3 features) ===")
print("Features: house size, bedrooms, age")
multi_model = MultipleLinearRegression(n_features=3, learning_rate=0.01)
multi_model.fit(X_scaled, y_scaled, epochs=1000, print_every=200)

print(f"\nWeights (standardized): {[round(w, 4) for w in multi_model.weights]}")
print(f"Bias (standardized): {multi_model.bias:.4f}")
print(f"R-squared: {multi_model.r_squared(X_scaled, y_scaled):.4f}")


=== Multiple Linear Regression (3 features) ===
Features: house size, bedrooms, age
  Epoch    0 | Cost: 0.9582
  Epoch  200 | Cost: 0.1195
  Epoch  400 | Cost: 0.1193
  Epoch  600 | Cost: 0.1193
  Epoch  800 | Cost: 0.1193

Weights (standardized): [0.7716, 0.2773, -0.2739]
Bias (standardized): 0.0000
R-squared: 0.8807


In [15]:
class PolynomialRegression:
    def __init__(self, degree, learning_rate=0.01):
        self.degree = degree
        self.weights = [0.0] * degree
        self.bias = 0.0
        self.lr = learning_rate

    def make_features(self, X):
        return [[x ** (d + 1) for d in range(self.degree)] for x in X]

    def predict(self, X):
        features = self.make_features(X)
        return [sum(w * f for w, f in zip(self.weights, row)) + self.bias for row in features]

    def fit(self, X, y, epochs=1000, print_every=200):
        features = self.make_features(X)
        n = len(y)
        for epoch in range(epochs):
            predictions = [sum(w * f for w, f in zip(self.weights, row)) + self.bias for row in features]
            errors = [pred - actual for pred, actual in zip(predictions, y)]
            for j in range(self.degree):
                grad = (2 / n) * sum(errors[i] * features[i][j] for i in range(n))
                self.weights[j] -= self.lr * grad
            grad_b = (2 / n) * sum(errors)
            self.bias -= self.lr * grad_b
            if epoch % print_every == 0:
                cost = sum(e ** 2 for e in errors) / n
                print(f"  Epoch {epoch:4d} | Cost: {cost:.6f}")
        return self

    def r_squared(self, X, y):
        predictions = self.predict(X)
        y_mean = sum(y) / len(y)
        ss_res = sum((actual - pred) ** 2 for actual, pred in zip(y, predictions))
        ss_tot = sum((actual - y_mean) ** 2 for actual in y)
        return 1 - (ss_res / ss_tot)


random.seed(42)
X_poly = [x / 10.0 for x in range(0, 50)]
y_poly = [0.5 * x ** 2 - 2 * x + 3 + random.gauss(0, 1.0) for x in X_poly]

x_max = max(abs(x) for x in X_poly)
X_poly_norm = [x / x_max for x in X_poly]
y_poly_mean = sum(y_poly) / len(y_poly)
y_poly_std = (sum((yi - y_poly_mean) ** 2 for yi in y_poly) / len(y_poly)) ** 0.5
y_poly_norm = [(yi - y_poly_mean) / y_poly_std for yi in y_poly]

print("\n=== Polynomial Regression (degree 2 vs degree 5) ===")
print("True relationship: y = 0.5x^2 - 2x + 3")

print("\nDegree 2:")
poly2 = PolynomialRegression(degree=2, learning_rate=0.1)
poly2.fit(X_poly_norm, y_poly_norm, epochs=2000, print_every=500)
print(f"  R-squared: {poly2.r_squared(X_poly_norm, y_poly_norm):.4f}")

print("\nDegree 5:")
poly5 = PolynomialRegression(degree=5, learning_rate=0.1)
poly5.fit(X_poly_norm, y_poly_norm, epochs=2000, print_every=500)
print(f"  R-squared: {poly5.r_squared(X_poly_norm, y_poly_norm):.4f}")

print("\nDegree 2 fits the true curve well. Degree 5 fits training data slightly better")
print("but risks overfitting on new data.")


=== Polynomial Regression (degree 2 vs degree 5) ===
True relationship: y = 0.5x^2 - 2x + 3

Degree 2:
  Epoch    0 | Cost: 1.000000
  Epoch  500 | Cost: 0.535618
  Epoch 1000 | Cost: 0.441336
  Epoch 1500 | Cost: 0.388633
  R-squared: 0.6408

Degree 5:
  Epoch    0 | Cost: 1.000000
  Epoch  500 | Cost: 0.345947
  Epoch 1000 | Cost: 0.336843
  Epoch 1500 | Cost: 0.333835
  R-squared: 0.6686

Degree 2 fits the true curve well. Degree 5 fits training data slightly better
but risks overfitting on new data.


In [22]:
class LassoRegression:
    def __init__(self, n_features, learning_rate=0.01, alpha=1.0):
        self.weights = [0.0] * n_features
        self.bias = 0.0
        self.lr = learning_rate
        self.alpha = alpha

    def predict_single(self, x):
        return sum(w * xi for w, xi in zip(self.weights, x)) + self.bias

    def predict(self, X):
        return [self.predict_single(x) for x in X]

    def fit(self, X, y, epochs=1000, print_every=200):
        n = len(y)
        n_features = len(X[0])
        for epoch in range(epochs):
            predictions = self.predict(X)
            errors = [pred - actual for pred, actual in zip(predictions, y)]
            mse = sum(e ** 2 for e in errors) / n
            reg_term = self.alpha * sum(w for w in self.weights)
            cost = mse + reg_term
            for j in range(n_features):
                grad = (2 / n) * sum(errors[i] * X[i][j] for i in range(n))
                grad += 2 * self.alpha * self.weights[j]
                self.weights[j] -= self.lr * grad
            grad_b = (2 / n) * sum(errors)
            self.bias -= self.lr * grad_b
            if epoch % print_every == 0:
                print(f"  Epoch {epoch:4d} | Cost: {cost:.4f} | L2 penalty: {reg_term:.4f}")
        return self


print("\n=== Lasso Regression (L1 Regularization) ===")
print("Same data as multiple regression, with alpha=0.1")
ridge = RidgeRegression(n_features=3, learning_rate=0.01, alpha=0.03)
ridge.fit(X_scaled, y_scaled, epochs=1000, print_every=200)
print(f"\nLasso weights: {[round(w, 4) for w in ridge.weights]}")
print(f"Plain weights: {[round(w, 4) for w in multi_model.weights]}")
print("Lasso weights are smaller (can be zero) due to the L1 penalty.")


=== Lasso Regression (L1 Regularization) ===
Same data as multiple regression, with alpha=0.1
  Epoch    0 | Cost: 1.0000 | L2 penalty: 0.0000
  Epoch  200 | Cost: 0.1413 | L2 penalty: 0.0209
  Epoch  400 | Cost: 0.1412 | L2 penalty: 0.0213
  Epoch  600 | Cost: 0.1412 | L2 penalty: 0.0213
  Epoch  800 | Cost: 0.1412 | L2 penalty: 0.0213

Lasso weights: [0.7503, 0.2735, -0.2682]
Plain weights: [0.7716, 0.2773, -0.2739]
Lasso weights are smaller (can be zero) due to the L1 penalty.


In [16]:
class RidgeRegression:
    def __init__(self, n_features, learning_rate=0.01, alpha=1.0):
        self.weights = [0.0] * n_features
        self.bias = 0.0
        self.lr = learning_rate
        self.alpha = alpha

    def predict_single(self, x):
        return sum(w * xi for w, xi in zip(self.weights, x)) + self.bias

    def predict(self, X):
        return [self.predict_single(x) for x in X]

    def fit(self, X, y, epochs=1000, print_every=200):
        n = len(y)
        n_features = len(X[0])
        for epoch in range(epochs):
            predictions = self.predict(X)
            errors = [pred - actual for pred, actual in zip(predictions, y)]
            mse = sum(e ** 2 for e in errors) / n
            reg_term = self.alpha * sum(w ** 2 for w in self.weights)
            cost = mse + reg_term
            for j in range(n_features):
                grad = (2 / n) * sum(errors[i] * X[i][j] for i in range(n))
                grad += 2 * self.alpha * self.weights[j]
                self.weights[j] -= self.lr * grad
            grad_b = (2 / n) * sum(errors)
            self.bias -= self.lr * grad_b
            if epoch % print_every == 0:
                print(f"  Epoch {epoch:4d} | Cost: {cost:.4f} | L2 penalty: {reg_term:.4f}")
        return self


print("\n=== Ridge Regression (L2 Regularization) ===")
print("Same data as multiple regression, with alpha=0.1")
ridge = RidgeRegression(n_features=3, learning_rate=0.01, alpha=0.1)
ridge.fit(X_scaled, y_scaled, epochs=1000, print_every=200)
print(f"\nRidge weights: {[round(w, 4) for w in ridge.weights]}")
print(f"Plain weights: {[round(w, 4) for w in multi_model.weights]}")
print("Ridge weights are smaller (shrunk toward zero) due to the L2 penalty.")


=== Ridge Regression (L2 Regularization) ===
Same data as multiple regression, with alpha=0.1
  Epoch    0 | Cost: 1.0000 | L2 penalty: 0.0000
  Epoch  200 | Cost: 0.1881 | L2 penalty: 0.0624
  Epoch  400 | Cost: 0.1881 | L2 penalty: 0.0633
  Epoch  600 | Cost: 0.1881 | L2 penalty: 0.0633
  Epoch  800 | Cost: 0.1881 | L2 penalty: 0.0633

Ridge weights: [0.7052, 0.2647, -0.2558]
Plain weights: [0.7716, 0.2773, -0.2739]
Ridge weights are smaller (shrunk toward zero) due to the L2 penalty.


In [17]:
from sklearn.linear_model import LinearRegression as SklearnLR
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

np.random.seed(42)
X_sk = np.random.uniform(0, 10, (100, 1))
y_sk = 3.0 * X_sk.squeeze() + 7.0 + np.random.normal(0, 2.0, 100)

X_train, X_test, y_train, y_test = train_test_split(X_sk, y_sk, test_size=0.2, random_state=42)

lr = SklearnLR()
lr.fit(X_train, y_train)
y_pred = lr.predict(X_test)

print("=== Scikit-learn Linear Regression ===")
print(f"Coefficient (w): {lr.coef_[0]:.4f}")
print(f"Intercept (b): {lr.intercept_:.4f}")
print(f"R-squared (test): {r2_score(y_test, y_pred):.4f}")
print(f"MSE (test): {mean_squared_error(y_test, y_pred):.4f}")

poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly_sk = poly.fit_transform(X_train)
X_poly_test = poly.transform(X_test)

lr_poly = SklearnLR()
lr_poly.fit(X_poly_sk, y_train)
print(f"\nPolynomial degree 2 R-squared: {r2_score(y_test, lr_poly.predict(X_poly_test)):.4f}")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)
print(f"Ridge R-squared: {r2_score(y_test, ridge.predict(X_test_scaled)):.4f}")
print(f"Ridge coefficient: {ridge.coef_[0]:.4f}")

=== Scikit-learn Linear Regression ===
Coefficient (w): 2.9197
Intercept (b): 7.2858
R-squared (test): 0.9686
MSE (test): 2.6148

Polynomial degree 2 R-squared: 0.9695
Ridge R-squared: 0.9689
Ridge coefficient: 8.3983
